In [7]:
import json
import os
from urllib.error import URLError
from urllib.parse import urlencode
from urllib.request import urlopen

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_deepseek import ChatDeepSeek
from rich import print as rprint

load_dotenv(override=True)

api_key = os.getenv("DEEPSEEK_API_KEY")
api_base = os.getenv("DEEPSEEK_API_BASE")

### 模型初始化的参数

In [2]:

chat_deepseek =  ChatDeepSeek(
    model="deepseek-v4-flash",    # model_name='deepseek-flush', # langchain模型并没有声明deepseek-flush的模型信息
)

rprint(chat_deepseek.profile)

{
    'name': 'DeepSeek V4 Flash',
    'release_date': '2026-04-24',
    'last_updated': '2026-04-24',
    'open_weights': True,
    'max_input_tokens': 1000000,
    'max_output_tokens': 384000,
    'text_inputs': True,
    'image_inputs': False,
    'audio_inputs': False,
    'video_inputs': False,
    'text_outputs': True,
    'image_outputs': False,
    'audio_outputs': False,
    'video_outputs': False,
    'reasoning_output': True,
    'tool_calling': True,
    'structured_output': True,
    'attachment': False,
    'temperature': True
}

### init_chat_model的模型参数

In [4]:
chat_model = init_chat_model(api_key = api_key,api_base =  api_base,model="deepseek-v4-flash")

rprint(chat_model.profile)

{
    'name': 'DeepSeek V4 Flash',
    'release_date': '2026-04-24',
    'last_updated': '2026-04-24',
    'open_weights': True,
    'max_input_tokens': 1000000,
    'max_output_tokens': 384000,
    'text_inputs': True,
    'image_inputs': False,
    'audio_inputs': False,
    'video_inputs': False,
    'text_outputs': True,
    'image_outputs': False,
    'audio_outputs': False,
    'video_outputs': False,
    'reasoning_output': True,
    'tool_calling': True,
    'structured_output': True,
    'attachment': False,
    'temperature': True
}

### model_kwargs  透传模型支持，但是langchain没有列出的字段(标准OpenAI的API参数)

In [9]:
@tool
def get_weather(city: str) -> str:
    """获取指定城市的实时天气。"""
    try:
        location_query = urlencode({"name": city, "count": 1, "language": "zh", "format": "json"})
        with urlopen(f"https://geocoding-api.open-meteo.com/v1/search?{location_query}", timeout=10) as response:
            location_data = json.load(response)

        locations = location_data.get("results", [])
        if not locations:
            return f"没有找到城市：{city}"

        location = locations[0]
        weather_query = urlencode({
            "latitude": location["latitude"],
            "longitude": location["longitude"],
            "current": "temperature_2m,apparent_temperature,relative_humidity_2m,weather_code,wind_speed_10m",
            "timezone": "auto",
        })
        with urlopen(f"https://api.open-meteo.com/v1/forecast?{weather_query}", timeout=10) as response:
            weather_data = json.load(response)

        return json.dumps({
            "城市": location["name"],
            "地区": location.get("admin1"),
            "国家": location.get("country"),
            "当前天气": weather_data["current"],
            "单位": weather_data["current_units"],
        }, ensure_ascii=False)
    except (URLError, TimeoutError, json.JSONDecodeError, KeyError) as exc:
        return f"天气服务请求失败：{exc}"


weather_tool_schema = convert_to_openai_tool(get_weather)
chat_deepseek = ChatDeepSeek(
    model="deepseek-chat",
    model_kwargs={"tools": [weather_tool_schema]},
)

messages = [("user", "今天上海天气如何？")]
response = chat_deepseek.invoke(messages)
rprint(response.tool_calls)


AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 38,
            'prompt_tokens': 270,
            'total_tokens': 308,
            'completion_tokens_details': None,
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 128,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 128,
            'prompt_cache_miss_tokens': 142
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': '20d03b82-7208-4fa9-b073-30f05f95b1b6',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a0c33c-79e6-7123-bf14-f63dc5b0779b-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '上海'},
            'id': 'call_00_eBAH9hWI0SGQcm27On8p6032',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 270,
        'output_tokens': 38,
        'total_tokens': 308,
        'input_token_details': {'cache_read': 128},
        'output_token_details': {}
    }
)

[{'name': 'get_weather', 'args': {'city': '上海'}, 'id': 'call_00_eBAH9hWI0SGQcm27On8p6032', 'type': 'tool_call'}]

### extra_body
* 非openai，而是提供商特有的非标准参数（ DeepSeek 的 thinking）

In [12]:
chat_deepseek = ChatDeepSeek(
    model="deepseek-chat",
    extra_body={"thinking": {'type':'enabled'}},
)
response = chat_deepseek.invoke('你是谁，一句话回答')
rprint(response)  # 包含思考过程additional_kwargs.reasoning_content

AIMessage(
    content='我是人工智能助手，可以回答你的问题。',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 
'我们需要回答用户中文：“你是谁，一句话回答”。要求一句话。需要身份？我是AI助手，由OpenAI？ 
现在系统说我是AI助手？我们应该是 DeepSeek？ 等等，需要知道身份。系统未明确模型名，仅“AI assistant accessed via an 
API.” 
用户问“你是谁，一句话回答”。应一句话回答。可以：“我是由OpenAI提供的人工智能助手。”但如果是通过API，未必OpenAI？ 
实际上系统没有说 
OpenAI。常见应回答“我是人工智能助手”。一句话即可。要严格一句话。可以用“我是由人工智能技术驱动的助手，可以回答你的问
题。” 确保一句话。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 143,
            'prompt_tokens': 34,
            'total_tokens': 177,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 133,
                'rejected_prediction_tokens': None,
                'text_tokens': None
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 0,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 34
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': '89d46859-c5a1-4b9f-bf72-5bf598483e36',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a0c345-4cda-7590-9092-73da6c1f7450-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 34,
        'output_tokens': 143,
        'total_tokens': 177,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 133}
    }
)

### config  指定调用时的参数

In [ ]:
chat_model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_base=api_base,
    api_key=api_key,
    max_tokens=100,
    temperature=0.2,
    configurable_fields=['max_token', 'temperature'] # 允许config修改的配置
)

chat_model.invoke('简短介绍下自己')